In [1]:
import os
import pickle
import torch
from torch.utils.data import Dataset
import random

AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20  # padding 또는 unknown
}

def random_voxel_rotate(voxel):
    # voxel: Tensor [C, D, H, W]
    if random.random() < 0.5:  # 50% 확률로 회전 적용
        axes = [(2, 3), (1, 3), (1, 2)]  # (H, W), (D, W), (D, H)
        k = random.choice([1, 2, 3])  # 실제 회전만 (0 제외)
        axis = random.choice(axes)
        voxel = torch.rot90(voxel, k=k, dims=axis)
    return voxel

def random_voxel_flip(voxel):
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[1])  # D-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[2])  # H-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[3])  # W-axis flip
    return voxel

class VoxelDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, aug=False):
        self.df = df.reset_index(drop=True)
        self.voxel_cache_dir = voxel_cache_dir
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["UniProtID"]
        mut_pos = row["MutPos(pdb)"]
        wt = row["WT"]
        mut = row["Mut"]
        label = row["Label"]

        key = f"{uid}_{mut_pos}"
        voxel_path = os.path.join(self.voxel_cache_dir, f"{key}.pkl")

        # Load voxel
        with open(voxel_path, "rb") as f:
            data = pickle.load(f)
            feature = data["feature"]  # shape: (1, 7, 7, 7, 63)

        # Preprocess
        feature_tensor = torch.from_numpy(feature).permute(0, 4, 1, 2, 3).float().squeeze(0)  # (63, 7, 7, 7)

        if self.aug:
            feature_tensor = random_voxel_rotate(feature_tensor)
            feature_tensor = random_voxel_flip(feature_tensor)

        # Convert WT/Mut AA to index
        ref_idx = torch.tensor(AA_TO_INDEX.get(str(wt), 20), dtype=torch.long)
        mut_idx = torch.tensor(AA_TO_INDEX.get(str(mut), 20), dtype=torch.long)

        return feature_tensor, ref_idx, mut_idx, torch.tensor(label).long()
    
class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61, aug=False):
        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df = df
        self.max_depth = max_depth
        self.win_size = win_size
        self.half_win = win_size // 2  # 중심에서 양쪽 길이
        self.aug = aug
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["UniProtID"]
        mut_pos = int(row["MutPos"]) - 1  # 1-based → 0-based
        label = int(row["Label"])
        mut = row["Mut"].upper()

        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]  # 보통 첫 줄이 ref seq

        # 변이 반영된 mut_seq 생성
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq):
            mut_seq[mut_pos] = mut

        # 사용될 시퀀스: 변이 시퀀스 + ref 시퀀스 + MSA
        seqs_to_use = [mut_seq, list(query_seq)]
        if len(msa_seqs) > 1:
            seqs_to_use += [list(seq) for seq in msa_seqs[1:self.max_depth - 2]]

        if self.aug:
            msa_part = seqs_to_use[2:]  # 변이+ref 제외
            random.shuffle(msa_part)   # 순서 섞기
            seqs_to_use = seqs_to_use[:2] + msa_part

        centered_msa = []
        for seq in seqs_to_use:
            window = []
            for i in range(self.win_size):
                seq_idx = mut_pos - self.half_win + i
                if 0 <= seq_idx < len(seq):
                    aa = seq[seq_idx]
                else:
                    aa = '-'
                window.append(AA_TO_INDEX.get(aa, 20))
            centered_msa.append(window)

        # depth padding
        while len(centered_msa) < self.max_depth:
            centered_msa.append([20] * self.win_size)  # 20은 패딩 인덱스

        msa_tensor = torch.tensor(centered_msa[:self.max_depth]).long()  # [D, L]
        msa_tensor = msa_tensor.transpose(0, 1)  # [L, D]

        return {
            "msa": msa_tensor,  # [L, D]
            "label": torch.tensor(label).long()
        }

class MultimodalDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, msa_dict_path, 
                 voxel_aug=False, msa_aug=False, max_depth=80, win_size=61):
        self.df = df.reset_index(drop=True)
        self.voxel_dataset = VoxelDataset(df, voxel_cache_dir, aug=voxel_aug)
        self.msa_dataset = MSADataset(df, msa_dict_path, max_depth=max_depth, win_size=win_size, aug=msa_aug)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        voxel_feat, ref_idx, mut_idx, label = self.voxel_dataset[idx]
        msa_data = self.msa_dataset[idx]  # returns dict with "msa", "label"
        msa_tensor = msa_data["msa"]
        
        # 라벨 일치 확인 (안전용)
        assert label == msa_data["label"], "Mismatch in label!"

        return {
            "voxel": voxel_feat,      # [63, 7, 7, 7]
            "ref_idx": ref_idx,
            "mut_idx": mut_idx,
            "msa": msa_tensor,        # [L=61, D]
            "label": label
        }

In [2]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba
import torch.nn.functional as F
import numpy as np

class VoxelMBConvClassifier(nn.Module):
    def __init__(self, in_ch=63, emb_dim=128, dropout_p=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            MBConv3D(in_ch, 64, expand_ratio=6),
            MBConv3D(64, 64, expand_ratio=6),
            MBConv3D(64, emb_dim, expand_ratio=6)
        )
        self.pool = nn.AdaptiveAvgPool3d(1)  # → [B, 128, 1, 1, 1]
        self.classifier = nn.Sequential(
            nn.Flatten(),                             # → [B, 128]
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(emb_dim, 1),
            nn.Sigmoid()  # Binary classification
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(-1)


class SqueezeExcitation3D(nn.Module):
    def __init__(self, in_channels, reduction=24):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.se = nn.Sequential(
            nn.Conv3d(in_channels, in_channels // reduction, kernel_size=1),
            nn.SiLU(),
            nn.Conv3d(in_channels // reduction, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.se(self.pool(x))
        return x * scale

class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=6, kernel_size=3, stride=1, se_reduction=24):
        super().__init__()
        mid_ch = in_ch * expand_ratio

        self.use_res_connect = (stride == 1 and in_ch == out_ch)

        self.expand = nn.Sequential(
            nn.Conv3d(in_ch, mid_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        ) if expand_ratio != 1 else nn.Identity()

        self.depthwise = nn.Sequential(
            nn.Conv3d(mid_ch, mid_ch, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size//2, groups=mid_ch, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        )

        self.se = SqueezeExcitation3D(mid_ch, reduction=se_reduction)

        self.project = nn.Sequential(
            nn.Conv3d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        identity = x
        out = self.expand(x)
        out = self.depthwise(out)
        out = self.se(out)
        out = self.project(out)

        if self.use_res_connect:
            return out + identity
        else:
            return out

# --- Input Embedding ---
class MSAInputEmbedding(nn.Module):
    def __init__(self, vocab_size=21, dim=256):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=dim)

    def forward(self, x):  # x: (B, L, D)
        return self.embedding(x)  # → (B, L, D, C)

# --- MambaRMSNorm ---
class MambaRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) / (x.shape[-1] ** 0.5)
        return self.weight * x / (norm + self.eps)

# --- Cross-Axial Mamba Block ---
class CrossAxialMambaMSA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm_L = MambaRMSNorm(dim)
        self.norm_D = MambaRMSNorm(dim)

        self.mamba_L = Mamba(d_model=dim, expand=1)
        self.conv_D = nn.Conv1d(in_channels=dim, out_channels=dim, kernel_size=5, padding=2)

    def forward(self, x):  # x: (B, L, D, C)
        B, L, D, C = x.shape
        center_L = L // 2  # = 30

        # --- D-axis: only at center L position ---
        x_d_center = self.norm_D(x[:, center_L])  # (B, D, C)
        x_d = x_d_center.transpose(1, 2)          # (B, C, D)
        d_out = self.conv_D(x_d).transpose(1, 2).unsqueeze(1)  # (B, 1, D, C)

        # --- L-axis: full Mamba ---
        x_l = self.norm_L(x).permute(0, 2, 1, 3).contiguous().view(B * D, L, C)
        l_out = self.mamba_L(x_l).view(B, D, L, C).permute(0, 2, 1, 3)  # (B, L, D, C)

        # --- Residual ---
        # d_out is only for center, rest is zero
        d_full = torch.zeros_like(x)
        d_full[:, center_L:center_L+1] = d_out

        return x + d_full + l_out
    
# # --- Encoder ---
class MSAEncoder(nn.Module):
    def __init__(self, num_layers=8, dim=256):
        super().__init__()
        self.embeddings = MSAInputEmbedding(dim=dim)
        self.blocks = nn.ModuleList([
            CrossAxialMambaMSA(dim) for _ in range(num_layers)
        ])
        self.norm_f = MambaRMSNorm(dim)

    def forward(self, x):  # x: (B, L, D)
        x = self.embeddings(x)  # → (B, L, D, C)
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x)  # (B, L, D, C)



class VoxelBranch(nn.Module):
    def __init__(self, in_ch=63, emb_dim=128):
        super().__init__()
        
        # 3D 구조 백본
        self.backbone = nn.Sequential(
            MBConv3D(in_ch, 32, expand_ratio=6),     # [7×7×7]
            MBConv3D(32, 32, expand_ratio=6),
            MBConv3D(32, 48, expand_ratio=6),
            MBConv3D(48, 48, expand_ratio=6),
            MBConv3D(48, 64, expand_ratio=6),
            MBConv3D(64, 64, expand_ratio=6, stride=2),  # 다운샘플링: → [4×4×4]
            MBConv3D(64, 64, expand_ratio=6),
            MBConv3D(64, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6)
        )
        self.pool = nn.AdaptiveAvgPool3d(1)  # → [B, 128, 1, 1, 1]

        # Mutation Embedding (64 + 64 → 128)
        self.ref_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_fusion = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU()
        )

        self.refine = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU()
        )

    def forward(self, x, ref_idx, mut_idx):
        x = self.backbone(x)               # [B, 128, 7, 7, 7]
        x = self.pool(x).squeeze(-1).squeeze(-1).squeeze(-1)  # → [B, 128]

        # Mutation embedding
        ref_vec = self.ref_emb(ref_idx)    # [B, 64]
        mut_vec = self.mut_emb(mut_idx)    # [B, 64]
        mut_feat = self.mut_fusion(torch.cat([ref_vec, mut_vec], dim=1))  # [B, 128]

        # Combine structure & mutation features
        x = x + mut_feat                   # [B, 128]

        return self.refine(x)       # [B, 128]
    
class MSABranch(nn.Module):
    def __init__(self, num_layers=4, dim=128):
        super().__init__()

        self.encoder = MSAEncoder(num_layers=num_layers, dim=dim)

        self.refine = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU()
        )

    def forward(self, x):  # x: (B, L, D, C)
        x = self.encoder(x) 
        
        center_L = x.shape[1] // 2   # 30
        x = x[:, center_L]           # (B, D, C)
        x = x.mean(dim=1)            # (B, C)
        return self.refine(x)        # (B, C=128)

class EvoStructCLIP(nn.Module):
    def __init__(self, voxel_ch=63, mb_layers=8, embed_dim=128, use_concat=True, dropout_p=0.3):
        super().__init__()
        self.voxel_encoder = VoxelBranch(in_ch=voxel_ch, emb_dim=embed_dim)
        self.msa_encoder = MSABranch(num_layers=mb_layers, dim=embed_dim)
        self.use_concat = use_concat

        # log(1 / 0.07) ≈ 2.6592 → exp(logit_scale) ≈ 14.2857
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

        fused_dim = embed_dim * 2 if use_concat else embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.SiLU(),
            # nn.Dropout(dropout_p),
            nn.Linear(embed_dim, 1)
        )

    def _get_vector_norm(self, tensor: torch.Tensor) -> torch.Tensor:
        
        square_tensor = torch.pow(tensor, 2)
        sum_tensor = torch.sum(square_tensor, dim=-1, keepdim=True)
        normed_tensor = torch.pow(sum_tensor, 0.5)
        
        return normed_tensor

    def forward(self, voxel, ref_idx, mut_idx, msa):
        # Raw features
        voxel_feat = self.voxel_encoder(voxel, ref_idx, mut_idx)  # [B, 128]
        msa_feat = self.msa_encoder(msa)                          # [B, 128]

        voxel_embeds = voxel_feat / self._get_vector_norm(voxel_feat)
        msa_embeds = msa_feat / self._get_vector_norm(msa_feat)

        logits_per_voxel = torch.matmul(voxel_embeds, msa_embeds.t().to(voxel_embeds.device))
        logits_per_voxel = logits_per_voxel * self.logit_scale.exp().to(voxel_embeds.device)

        logits_per_msa = logits_per_voxel.t() 

        # For classification
        fused = torch.cat([voxel_feat, msa_feat], dim=-1) if self.use_concat else voxel_feat + msa_feat
        logits = self.classifier(fused)

        return {
            "logits": logits,
            "logits_per_msa": logits_per_msa,
            "logits_per_voxel": logits_per_voxel,
            "voxel_feat": voxel_feat,
            "msa_feat": msa_feat
        }

/home/kunny/miniconda3/envs/mamba_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import KFold

df = pd.read_csv(r"/mnt/c/Users/Kunny/Research/Project/BiConVarNet/filtered_variants_cleaned_final.tsv", sep="\t", )

# 10-fold 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
splits = list(kf.split(df))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# oversampling: label == 1
pos_df = train_df[train_df["Label"] == 1]
neg_df = train_df[train_df["Label"] == 0]

repeat_factor = max(1, len(neg_df) // max(len(pos_df), 1))
oversampled_train_df = pd.concat([neg_df, pd.concat([pos_df] * repeat_factor)], ignore_index=True)
oversampled_train_df = oversampled_train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # 셔플

from torch.utils.data import DataLoader
        
voxel_cache_dir = "/mnt/e/CAGI_data/voxel_cache_4"
msa_dict_path = "/mnt/e/CAGI_data/msa_dict_valid_new.pkl"

train_dataset = MultimodalDataset(oversampled_train_df, voxel_cache_dir, msa_dict_path, voxel_aug=False, msa_aug=False)
val_dataset   = MultimodalDataset(val_df, voxel_cache_dir, msa_dict_path)

train_loader = DataLoader(train_dataset, batch_size=80, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EvoStructCLIP(voxel_ch=48, mb_layers=6, embed_dim=128, use_concat=True, dropout_p=0.3).to(device)

# --- Binary classification (output: [B, 1]) + CLIP
criterion = nn.BCEWithLogitsLoss()  # sigmoid + BCE
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

num_epochs = 100
best_pr_auc = 0.0
save_path = "/mnt/e/CAGI_data/best_model_250827_clip.pth"

# --- Contrastive loss
def contrastive_loss(logits: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits, torch.arange(len(logits), device=logits.device))

def compute_clip_loss(similarity: torch.Tensor) -> torch.Tensor:
    return (contrastive_loss(similarity) + contrastive_loss(similarity.t())) / 2

def fusemix(voxel_feat, msa_feat, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(voxel_feat.size(0), device=voxel_feat.device)

    voxel_feat_shuffled = voxel_feat[idx]
    msa_feat_shuffled = msa_feat[idx]

    voxel_mix = lam * voxel_feat + (1 - lam) * voxel_feat_shuffled
    msa_mix = lam * msa_feat + (1 - lam) * msa_feat_shuffled

    return voxel_mix, msa_mix

# --- Train Loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        voxel = batch["voxel"].to(device)
        ref_idx = batch["ref_idx"].to(device)
        mut_idx = batch["mut_idx"].to(device)
        msa = batch["msa"].to(device)
        label = batch["label"].float().to(device)  # BCE → float

        optimizer.zero_grad()

        out = model(voxel, ref_idx, mut_idx, msa)
        logits = out["logits"].squeeze(-1)  # [B]
        cls_loss = criterion(logits, label)

        clip_loss_val = compute_clip_loss(out["logits_per_voxel"])

        voxel_mix, msa_mix = fusemix(out["voxel_feat"], out["msa_feat"])

        voxel_mix_norm = voxel_mix / voxel_mix.norm(dim=-1, keepdim=True)
        msa_mix_norm = msa_mix / msa_mix.norm(dim=-1, keepdim=True)
        logits_per_voxel_mix = torch.matmul(voxel_mix_norm, msa_mix_norm.T) * model.logit_scale.exp()
        loss_mix = compute_clip_loss(logits_per_voxel_mix)

        total_loss = cls_loss +    clip_loss_val + 0.7 * loss_mix

        total_loss.backward()
        optimizer.step()
        train_loss += total_loss.item() * voxel.size(0)

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            voxel = batch["voxel"].to(device)
            ref_idx = batch["ref_idx"].to(device)
            mut_idx = batch["mut_idx"].to(device)
            msa = batch["msa"].to(device)
            label = batch["label"].float().to(device)

            out = model(voxel, ref_idx, mut_idx, msa)
            logits = out["logits"].squeeze(-1)  # [B]
            loss = criterion(logits, label)

            probs = torch.sigmoid(logits)  # [B]

            val_loss += loss.item() * voxel.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    # --- Metric 계산 ---
    avg_val_loss = val_loss / len(val_loader.dataset)
    pr_auc = average_precision_score(all_labels, all_probs)
    roc_auc = roc_auc_score(all_labels, all_probs)

    # 0.5 기준 이진 분류
    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    acc = accuracy_score(all_labels, preds)

    # --- 출력 ---
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"Val PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f} | Accuracy: {acc:.4f}")

    if pr_auc > best_pr_auc:  
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), save_path)
        print(f">>> Best model saved! PR-AUC: {pr_auc:.4f}")
        

Epoch 1 [Val]: 100%|██████████| 962/962 [04:27<00:00,  3.60it/s]



Epoch 1/100
Train Loss: 1.6383 | Val Loss: 0.4017
Val PR-AUC: 0.8076 | ROC-AUC: 0.8824 | Accuracy: 0.8198
>>> Best model saved! PR-AUC: 0.8076


Epoch 2 [Val]: 100%|██████████| 962/962 [00:32<00:00, 29.17it/s]



Epoch 2/100
Train Loss: 1.0313 | Val Loss: 0.4098
Val PR-AUC: 0.8347 | ROC-AUC: 0.9033 | Accuracy: 0.8193
>>> Best model saved! PR-AUC: 0.8347


Epoch 3 [Val]: 100%|██████████| 962/962 [00:33<00:00, 29.06it/s]


...


Epoch 6/100
Train Loss: 0.6979 | Val Loss: 0.7452
Val PR-AUC: 0.8699 | ROC-AUC: 0.9257 | Accuracy: 0.7121
>>> Best model saved! PR-AUC: 0.8699


Epoch 7 [Val]: 100%|██████████| 962/962 [00:33<00:00, 28.60it/s]



Epoch 7/100
Train Loss: 0.5711 | Val Loss: 0.6943
Val PR-AUC: 0.8566 | ROC-AUC: 0.9094 | Accuracy: 0.7484


Epoch 8 [Train]:  80%|████████  | 1385/1731 [12:00<03:00,  1.92it/s]


KeyboardInterrupt: 